# Hypervolume benchmarks

A self-contained Python notebook for computing WFG hypervolume and comparing serial and threaded implementations. Run the setup cell, then choose one or more benchmark cells.

In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from math import prod, sqrt
from time import perf_counter
from typing import Iterable

import matplotlib.pyplot as plt

Point = list[float]
PointSet = list[Point]

class SimpleRng:
    def __init__(self, seed: int) -> None: self.state = seed
    def next_f64(self) -> float:
        self.state = (self.state * 6364136223846793005 + 1) & 0xFFFFFFFFFFFFFFFF
        return (self.state >> 33) / 2147483648.0
    def gen_index(self, upper: int) -> int: return 0 if upper <= 1 else int(self.next_f64() * upper) % upper
    def shuffle(self, values: list[object]) -> None:
        for i in range(len(values) - 1, 0, -1):
            j = self.gen_index(i + 1); values[i], values[j] = values[j], values[i]

@dataclass
class BenchmarkResult:
    dimension: int; num_points: int; hv: float; serial_time: float; parallel_time: float; serial_std: float; parallel_std: float

@dataclass
class WorstCaseBenchmarkResult:
    dimension: int; hv: float; serial_time: float; parallel_time: float

def dominates(p: Point, q: Point) -> bool:
    return all(pi >= qi for pi, qi in zip(p, q)) and any(pi > qi for pi, qi in zip(p, q))

def get_non_dominated(points: PointSet) -> PointSet:
    return [p[:] for i, p in enumerate(points) if not any(j != i and dominates(q, p) for j, q in enumerate(points))]

def limit_set(points: PointSet, limit: Point) -> PointSet:
    return [[min(a, b) for a, b in zip(point, limit)] for point in points]

def hypervolume_2d(points: PointSet, ref: Point) -> float:
    hv, previous_x = 0.0, ref[0]
    for point in sorted(points, key=lambda p: p[0]):
        hv += (point[0] - previous_x) * (point[1] - ref[1]); previous_x = point[0]
    return hv

def _wfg(points: PointSet, ref: Point) -> float:
    if not points: return 0.0
    if len(ref) == 1: return max(points[0][0] - ref[0], 0.0)
    if len(ref) == 2: return hypervolume_2d(points, ref)
    return sum(_exclusive(p, points[i + 1:], ref) for i, p in enumerate(points))

def _exclusive(point: Point, rest: PointSet, ref: Point) -> float:
    inclusive = prod(p - r for p, r in zip(point, ref))
    nd = get_non_dominated(limit_set(rest, point))
    return inclusive if not nd else inclusive - _wfg(nd, ref)

def _exclusive_parallel(point: Point, rest: PointSet, ref: Point, depth: int) -> float:
    inclusive = prod(p - r for p, r in zip(point, ref))
    nd = get_non_dominated(limit_set(rest, point))
    if not nd: return inclusive
    if depth <= 0: return inclusive - _wfg(nd, ref)
    with ThreadPoolExecutor() as pool:
        return inclusive - sum(pool.submit(_exclusive_parallel, p, nd[i + 1:], ref, depth - 1).result() for i, p in enumerate(nd))

def calculate_hv_serial(points: PointSet, ref: Point) -> float: return _wfg(points, ref)
def calculate_hv_parallel(points: PointSet, ref: Point, depth: int = 2) -> float:
    if depth <= 0: return _wfg(points, ref)
    with ThreadPoolExecutor() as pool:
        futures = [pool.submit(_exclusive_parallel, p, points[i + 1:], ref, depth - 1) for i, p in enumerate(points)]
        return sum(f.result() for f in futures)

def generate_non_dominated_dataset(n: int, dim: int) -> PointSet:
    total, rng, seen, points = 1000, SimpleRng(12345 + n + dim * 10_000), set(), []
    while len(points) < n:
        cuts = sorted({1 + rng.gen_index(total - 1) for _ in range(dim - 1)})
        if len(cuts) != dim - 1: continue
        coords, previous = [], 0
        for cut in cuts: coords.append(cut - previous); previous = cut
        coords.append(total - previous); rng.shuffle(coords)
        if tuple(coords) not in seen: seen.add(tuple(coords)); points.append([x / total for x in coords])
    rng.shuffle(points); return points

def generate_cyclic_points(dim: int) -> PointSet:
    return [[float((i + shift) % dim + 1) for i in range(dim)] for shift in range(dim)]

def mean_and_std(values: Iterable[float]) -> tuple[float, float]:
    values = list(values); mean = sum(values) / len(values)
    return mean, 0.0 if len(values) == 1 else sqrt(sum((v - mean) ** 2 for v in values) / (len(values) - 1))


In [ ]:
# Quick correctness check
points = [[0.2, 0.9], [0.5, 0.5], [0.9, 0.2]]
reference = [0.0, 0.0]
serial_hv = calculate_hv_serial(points, reference)
parallel_hv = calculate_hv_parallel(points, reference, depth=2)
assert abs(serial_hv - parallel_hv) < 1e-12
print(f'Hypervolume: {serial_hv:.4f} (serial and parallel agree)')

In [ ]:
def benchmark_random(dimensions, point_counts, runs=3, parallel_depth=2):
    results = []
    for dim in dimensions:
        for n in point_counts:
            points, ref = generate_non_dominated_dataset(n, dim), [0.0] * dim
            hv = calculate_hv_serial(points, ref)
            serial = []; parallel = []
            for _ in range(runs):
                start = perf_counter(); calculate_hv_serial(points, ref); serial.append((perf_counter() - start) * 1000)
                start = perf_counter(); calculate_hv_parallel(points, ref, parallel_depth); parallel.append((perf_counter() - start) * 1000)
            sm, ss = mean_and_std(serial); pm, ps = mean_and_std(parallel)
            results.append(BenchmarkResult(dim, n, hv, sm, pm, ss, ps))
            print(f'dim={dim:2}, n={n:3}, HV={hv:.3e}, serial={sm:.2f} ms, parallel={pm:.2f} ms')
    return results

# Keep this modest by default; expand these lists for the original full benchmark.
random_results = benchmark_random(dimensions=[2, 3, 4], point_counts=[20, 40, 60], runs=3, parallel_depth=2)

In [ ]:
def plot_random(results):
    for dim in sorted({r.dimension for r in results}):
        selected = [r for r in results if r.dimension == dim]
        plt.plot([r.num_points for r in selected], [r.serial_time for r in selected], 'o-', label=f'Serial, dim={dim}')
        plt.plot([r.num_points for r in selected], [r.parallel_time for r in selected], 's--', label=f'Parallel, dim={dim}')
    plt.xlabel('Number of points'); plt.ylabel('Mean time (ms)'); plt.title('Random non-dominated data')
    plt.legend(); plt.grid(alpha=.25); plt.show()

plot_random(random_results)

In [ ]:
def benchmark_worst_case(max_dimension=10, runs=3, parallel_depth=2):
    results = []
    for dim in range(2, max_dimension + 1):
        points, ref = generate_cyclic_points(dim), [0.0] * dim
        hv = calculate_hv_serial(points, ref)
        assert abs(hv - calculate_hv_parallel(points, ref, parallel_depth)) < max(abs(hv) * 1e-12, 1e-12)
        serial = []; parallel = []
        for _ in range(runs):
            start = perf_counter(); calculate_hv_serial(points, ref); serial.append((perf_counter() - start) * 1000)
            start = perf_counter(); calculate_hv_parallel(points, ref, parallel_depth); parallel.append((perf_counter() - start) * 1000)
        results.append(WorstCaseBenchmarkResult(dim, hv, sum(serial)/runs, sum(parallel)/runs))
    return results

worst_case_results = benchmark_worst_case()
plt.plot([r.dimension for r in worst_case_results], [r.serial_time for r in worst_case_results], 'o-', label='Serial')
plt.plot([r.dimension for r in worst_case_results], [r.parallel_time for r in worst_case_results], 's--', label='Parallel')
plt.xlabel('Dimension'); plt.ylabel('Mean time (ms)'); plt.title('Cyclic worst-case data'); plt.legend(); plt.grid(alpha=.25); plt.show()